# hazprec on Colab

Upload `hazard-precision.zip` with a top-level `hazard-precision/` directory to
`MyDrive/hazprec/`. Select a GPU with enough actual VRAM for the config, then run
these cells in order. High system RAM is not GPU VRAM. Failures raise exceptions;
the atlas requires a successful gate for this exact source/config/dependency set.


In [ ]:
from google.colab import drive
from pathlib import Path
import sys, os, subprocess, json
from zipfile import ZipFile

drive.mount('/content/drive')
RUNS = Path('/content/drive/MyDrive/hazprec/runs/colab_8b_v2')
RUNS.mkdir(parents=True, exist_ok=True)
with ZipFile('/content/drive/MyDrive/hazprec/hazard-precision.zip') as z:
    z.extractall('/content')
os.chdir('/content/hazard-precision')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[cuda,dev]'], check=True)
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], check=True)


In [ ]:
import yaml
from hazprec.config import load_config
from hazprec.utils.experiment import identity
from hazprec.theory import load_preregistration

CONFIG = RUNS / 'config.yaml'
if not CONFIG.exists():
    CONFIG.write_text(yaml.safe_dump(load_config('configs/colab_8b.yaml'), sort_keys=False))
cfg = load_config(CONFIG)

def run_logged(args, filename):
    with (RUNS / filename).open('a') as log:
        with subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as proc:
            for line in proc.stdout:
                print(line, end='')
                log.write(line)
                log.flush()
            status = proc.wait()
        if status:
            raise subprocess.CalledProcessError(status, args)

def cli(*args):
    return [sys.executable, '-m', 'hazprec.cli', *map(str, args)]


## 1. Tests and smoke
A smoke failure stops this cell. Investigate reference reliability before proceeding.

In [ ]:
run_logged([sys.executable, '-m', 'pytest', '-q', 'tests'], 'tests.txt')
run_logged(cli('smoke', '-c', CONFIG, '--hops', '1'), 'smoke.txt')


## 2. Readiness gate

This includes actual replay cost calibration. All five checks must pass.
If cost fails, choose a feasible config/budget and a new experiment directory;
do not silently waive the check. Existing successful reports are revalidated.


In [ ]:
GATE = RUNS / 'gate.json'

def require_gate():
    report = json.loads(GATE.read_text())
    if not report.get('ready') or report.get('experiment') != identity(load_config(CONFIG)):
        raise RuntimeError('A successful gate for this exact experiment is required')

if GATE.exists():
    previous = json.loads(GATE.read_text())
else:
    previous = {}
if not previous.get('ready') or previous.get('experiment') != identity(cfg):
    run_logged([sys.executable, 'scripts/phase1_gate.py', '-c', str(CONFIG),
                '--n', '20', '--out', str(GATE)], 'gate.txt')
require_gate()
(RUNS / 'requirements.txt').write_text(subprocess.check_output(
    [sys.executable, '-m', 'pip', 'freeze'], text=True))


## 3. Atlas and fit
Resume only the same experiment. Keep `.meta.json` sidecars with the data.

In [ ]:
require_gate()
ATLAS = RUNS / 'atlas.jsonl'
run_logged(cli('atlas', '-c', CONFIG, '--out', ATLAS), 'atlas.txt')
run_logged(cli('fit', '--atlas', ATLAS, '--out', RUNS / 'fit.json'), 'fit.txt')


## 4. Pre-register

Preserve/commit the resulting file and exact source before the sweep. The command
verifies atlas completeness and identity. An existing preregistration is never replaced.


In [ ]:
PRE = RUNS / 'prereg.json'
if PRE.exists():
    pre = load_preregistration(PRE)
    if pre.get('experiment') != identity(load_config(CONFIG)):
        raise RuntimeError('Existing preregistration belongs to another experiment')
else:
    run_logged(cli('preregister', '-c', CONFIG, '--atlas', ATLAS, '--out', PRE), 'preregister.txt')
print(PRE)


## 5. Paired evaluation
Run after preserving the preregistration. Analysis requires the full registered sweep.

In [ ]:
require_gate()
RUNFILE = RUNS / 'horizon.jsonl'
run_logged(cli('horizon', '-c', CONFIG, '--prereg', PRE, '--out', RUNFILE), 'horizon.txt')
run_logged(cli('analyze', '--runs', RUNFILE, '--prereg', PRE), 'analyze.txt')
